In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Leer datos de ventas 2024
# No subidos aún

In [ ]:
# path_archivo = "datos/Ventas por Cliente/Ventas 2025/2 Base Ventas x Facturas 2025 Abril - Mayo.xlsx"
# df = pd.read_excel(path_archivo, sheet_name=None, header=1)
# # Concatenamos todas las hojas en un solo dataframe
# df_concatenado = pd.concat(df.values(), ignore_index=True)

In [ ]:
# df_concatenado.head()

In [ ]:
sheets_por_archivo = {
    "6 Base Ventas x Facturas 2025 Noviembre.xlsx": 1,
    "1 Base Ventas x Facturas 2025 Enero - Marzo.xlsx": 3,
    "5 Base Ventas x Facturas 2025 Octubre.xlsx": 1,
    "3 Base Ventas x Facturas 2025 Junio.xlsx": 1,
    "2 Base Ventas x Facturas 2025 Abril - Mayo.xlsx": 2,
    "4 Base Ventas x Facturas 2025 Julio-Septiembre.xlsx": 3,
    "7 Base Ventas x Facturas 2025 Diciembre.xlsx": 1
}

In [ ]:
# Leer datos de ventas 2025
carpeta_ventas_2025 = "datos/Ventas por Cliente/Ventas 2025/"

# Armamos lista de dataframes para cada mes, luego los concatenamos en uno solo
dfs_2025 = {}

archivos_carpeta = [path for path in pd.io.common.os.listdir(carpeta_ventas_2025) if path.endswith(".xlsx")]
for archivo in archivos_carpeta:
    print(f"Cargando archivo: {archivo}")
    # Cargamos archivo en dataframe, un. dataframe por cada hoja independiente de sus nombres
    path_archivo = carpeta_ventas_2025 + archivo

    # Para cada archivo, revisar cuántas hojas tiene y cuál es el header correcto
    for n, hoja in enumerate(pd.ExcelFile(path_archivo).sheet_names):
        print(f"    Hoja: {hoja}")
        # Ver cual es el header correcto, probar header=0, si tiene mas de 2 unnamed, probar header=1
        if n == 0:
            header = 1
        else:
            header = 0

        df = pd.read_excel(path_archivo, sheet_name=hoja, header=header)
        print(f"        Columnas: {df.columns.tolist()}")
   
        llave = f"{archivo} - {hoja}"
        dfs_2025[llave] = df

In [ ]:
# Revisión de columnas y tipos de datos
for nombre_archivo, df in dfs_2025.items():
    print(f"Archivo: {nombre_archivo}")
    print("Columnas:", df.columns.tolist())
    print("Tipos de datos:\n", df.dtypes)
    print("-" * 40)

In [ ]:
# Estandarizar nombres/estructura y luego concatenar en una sola tabla
columnas_objetivo = [
    "Año", "Mes", "Cod Canal Comercial", "Clase Factura", "Fecha Factura",
    "Cod Cliente", "Nombre Cliente Padre", "Cod Consolidado", "Nombre Consolidado",
    "Nombre Familia", "N° Factura", "Nombre Marca", "Nombre Tipo Carne",
    "Cod SKU", "Nombre SKU", "Factura Venta", "Factura Kilos", "Kilo Real",
    "Kilos Nc", "Monto Nc", "Monto Real", "Precio"
]

equivalencias_columnas = {
    "Kilos Factura": "Factura Kilos",
    "Kilos Real": "Kilo Real",
    "Nº Factura": "N° Factura",
    "No Factura": "N° Factura"
}

columnas_enteras = [
    "Año", "Mes", "Fecha Factura", "Cod Cliente", "Cod Consolidado",
    "N° Factura", "Cod SKU", "Factura Venta", "Monto Real", "Monto Nc"
]
columnas_float = ["Factura Kilos", "Kilo Real", "Kilos Nc", "Precio"]
columnas_texto = [
    "Cod Canal Comercial", "Clase Factura", "Nombre Cliente Padre", "Nombre Consolidado",
    "Nombre Familia", "Nombre Marca", "Nombre Tipo Carne", "Nombre SKU"
]

def estandarizar_estructura(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    # Eliminar columnas índice residuales de Excel
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # Unificar nombres alternativos
    for col_origen, col_destino in equivalencias_columnas.items():
        if col_origen in df.columns:
            if col_destino in df.columns:
                df[col_destino] = df[col_destino].combine_first(df[col_origen])
            else:
                df[col_destino] = df[col_origen]
            df = df.drop(columns=[col_origen])

    # Completar faltantes para garantizar mismo esquema
    for col in columnas_objetivo:
        if col not in df.columns:
            df[col] = pd.NA

    # Conservar solo columnas objetivo y en orden
    df = df[columnas_objetivo]

    # Tipos consistentes
    for col in columnas_enteras:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in columnas_float:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in columnas_texto:
        df[col] = df[col].astype("string").str.strip()

    return df

dfs_2025_estandarizados = {k: estandarizar_estructura(v) for k, v in dfs_2025.items()}
ventas_2025 = pd.concat(dfs_2025_estandarizados.values(), ignore_index=True)
ventas_2025 = ventas_2025.sort_values(
    ["Año", "Mes", "Fecha Factura", "N° Factura", "Cod SKU"],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

print("Columnas finales:", ventas_2025.columns.tolist())
print("Tipos de datos finales:\n", ventas_2025.dtypes)
print("Total filas:", len(ventas_2025))

In [ ]:
# Exportar como csv
ventas_2025.to_csv("datos/Ventas por Cliente/ventas_2025_concatenado.csv", index=False)

In [ ]:
ventas_2025.columns

In [ ]:
ventas_2025.shape

In [ ]:
# Contar nans
ventas_2025.isna().sum()

In [ ]:
# Leer datos de ventas 2026
path_ventas_2026 = "datos/Ventas por Cliente/Ventas 2026/Base Ventas x Facturas 2026.xlsx"
ventas_2026 = pd.read_excel(path_ventas_2026, header=1)
ventas_2026.head()

In [ ]:
ventas_2026.columns

In [ ]:
# Calzar columnas 2025 con 2026 y unir
# pasar de
# Index(['Unnamed: 0', 'Año', 'Mes', 'Cod Canal Comercial', 'Clase Factura',
#        'Fecha Factura', 'Cod Cliente', 'Nombre Cliente Padre',
#        'Cod Consolidado', 'Nombre Consolidado', 'Nombre Familia', 'N° Factura',
#        'Nombre Marca', 'Nombre Tipo Carne', 'Cod SKU', 'Nombre SKU',
#        'Factura Venta', 'Factura Kilos', 'Kilo Real', 'Monto Real', 'Precio'],
#       dtype='str')
# A:
# Index(['Año', 'Mes', 'Cod Canal Comercial', 'Clase Factura', 'Fecha Factura',
#        'Cod Cliente', 'Nombre Cliente Padre', 'Cod Consolidado',
#        'Nombre Consolidado', 'Nombre Familia', 'N° Factura', 'Nombre Marca',
#        'Nombre Tipo Carne', 'Cod SKU', 'Nombre SKU', 'Factura Venta',
#        'Factura Kilos', 'Kilo Real', 'Kilos Nc', 'Monto Nc', 'Monto Real',
#        'Precio'],
#       dtype='str')
ventas_2026 = ventas_2026.rename(columns={
    "Unnamed: 0": "Índice",
    "Monto Real": "Monto Real",
    "Monto Nc": "Monto Nc",
    "Kilos Nc": "Kilos Nc"
})
ventas_2026 = ventas_2026.drop(columns=["Índice"])
ventas_2026.columns

In [ ]:
# unir ventas_2025 y ventas_2026
ventas = pd.concat([ventas_2025, ventas_2026], ignore_index=True)
ventas.shape

In [ ]:
ventas.to_csv("datos/Ventas por Cliente/ventas_2025_2026.csv", index=False)

In [ ]:
ventas_2026.to_csv("datos/Ventas por Cliente/ventas_2026.csv", index=False)